In [35]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains

from bs4 import BeautifulSoup

import pprint
import pandas as pd
import time

In [6]:
URL = "https://www.kyobobook.co.kr/"

In [66]:
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 5)

actions = ActionChains(driver)

# # # # # # # # # 설정 끝 # # # # # # # # #


# 드라이버 창 열기
driver.get(URL)


# 데이터 카테고리 확인 가능할 때까지 대기하기
category_button = wait.until(
  EC.element_to_be_clickable(
    (By.CSS_SELECTOR, "div#gnbWrap > div:first-child button")
  )
).click()

time.sleep(0.5)

category_nav_li = wait.until(
  EC.presence_of_all_elements_located(
    (By.CSS_SELECTOR, 'section[aria-label="카테고리"] > nav > ul > li')
  )
)

# .get_attribute("innerHTML")
# print(category_nav[0].text.split("\n"))
# ['국내도서', '문학', '취미/실용/여행', '생활/요리/건강', '예술/건축', '인문/....]

cat_datas = []
test_soup = ""
# 교보 Only는 제거하고 돌리기
for cat in category_nav_li[:-1]:
  actions.move_to_element(cat).perform()
  cat_data = cat.text.split("\n")

  # print(("href"))
  sub_cats_link = list(map(lambda e: e.get_attribute("href"), cat.find_elements(By.CSS_SELECTOR, "li > a")))

  # print(len(cat_data))
  # print(len(sub_cats_link))
  # print(cat_data[0])
  # print(sub_cats_link[0])
  cat_datas.append({
    cat_data[0]: {k: v for k, v in zip(cat_data[1:], sub_cats_link[1:])}
  })

  # 반복 없이
  driver.get(sub_cats_link[0])
  print(sub_cats_link[0], "탐색")
  
  time.sleep(1)
  for i in range(3):
    actions.scroll_by_amount(0, 4000).perform()
    time.sleep(0.5)
  time.sleep(1)
  print("출력합니다.")

  searched_books = driver.find_elements(
    By.XPATH, 
    '''
    //div[
      count(.//a)=2
      and .//strong[
        string-length(normalize-space(.)) > 0
        and translate(normalize-space(.), "0123456789,", "") = ""
        and following-sibling::*[1][
          self::strong
          and normalize-space(.)="원"
        ]
      ]
    ]
    '''
  )

  print(f"탐색한 책 div 패턴 개수: {len(searched_books)}")
  for book in searched_books:
    print(book.find_element(By.CSS_SELECTOR, "a").get_attribute("href"))



  # 테스트용 1번 실행 (국내도서만)
  break

print("\n --- --- \n")
pprint.pprint(cat_datas)

https://store.kyobobook.co.kr/category/domestic 탐색
출력합니다.
탐색한 책 div 패턴 개수: 127
https://product.kyobobook.co.kr/detail/S000220166821
https://product.kyobobook.co.kr/detail/S000220240950
https://product.kyobobook.co.kr/detail/S000220240828
https://product.kyobobook.co.kr/detail/S000220270663
https://product.kyobobook.co.kr/detail/S000220270663
https://product.kyobobook.co.kr/detail/S000220341532
https://product.kyobobook.co.kr/detail/S000220119827
https://product.kyobobook.co.kr/detail/S000220119415
https://product.kyobobook.co.kr/detail/S000220119415
https://product.kyobobook.co.kr/detail/S000000620195
https://product.kyobobook.co.kr/detail/S000000620195
https://product.kyobobook.co.kr/detail/S000220342469
https://product.kyobobook.co.kr/detail/S000220342469
https://product.kyobobook.co.kr/detail/S000220272256
https://product.kyobobook.co.kr/detail/S000220272256
https://product.kyobobook.co.kr/detail/S000220216599
https://product.kyobobook.co.kr/detail/S000220216599
https://product.kyob

In [40]:
soup = BeautifulSoup(test_soup, "html.parser")
print(soup.prettify())

<a class="bg-white font-medium px-7.5 h-[50px] flex items-center justify-between fz-16 w-full cursor-pointer focus:bg-white focus:font-medium" href="https://store.kyobobook.co.kr/category/kyobo-only">
 <div class="flex items-center gap-1">
  <img alt="교보Only" class="h-6 w-6" src="https://contents.kyobobook.co.kr/display/4_6ea5e213bf7347d9953337566ec78312.png"/>
  교보Only
 </div>
 <svg aria-hidden="true" class="block" fill="none" height="12" stroke="black" viewbox="0 0 12 12" width="12" xmlns="http://www.w3.org/2000/svg">
  <title>
   오른쪽 화살표 아이콘
  </title>
  <desc>
   오른쪽 화살표 아이콘
  </desc>
  <g clip-path="url(#clip0_10960_1586)">
   <path d="M4.125 10.125L8.0475 6.2025L4.17 2.3175" stroke="current" stroke-linecap="round" stroke-linejoin="round">
   </path>
  </g>
  <defs>
   <clippath id="clip0_10960_1586">
    <rect fill="white" height="12" width="12">
    </rect>
   </clippath>
  </defs>
 </svg>
</a>
<div aria-label="교보Only 메뉴" class="w-50 py-7.5 kds-scrollbar-slim absolute left-[300p